# Household energy prediction with a decision tree
Open this notebook from the repository folder and run the cells in order. The implementation in `decision_tree.py` uses the project's `energydata.csv`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image
from decision_tree import DEFAULT_OUTPUT_DIR, run_project
%matplotlib inline


## 1. Train and evaluate
This runs training-only exploration, time-based tuning, a baseline comparison, and final held-out evaluation.


In [ ]:
results, details = run_project()
display(results)
details


## 2. Inspect plots and demand errors

High demand starts at the 90th percentile of training usage. Mean signed error is predicted minus actual usage. Negative values indicate underprediction on average.

In [ ]:
for name in ["daily_usage", "hourly_usage", "actual_vs_predicted", "feature_importance", "tree_preview"]:
    display(Image(filename=str(DEFAULT_OUTPUT_DIR / f"{name}.png")))
display(pd.read_csv(DEFAULT_OUTPUT_DIR / "high_demand_errors.csv"))


## 3. Results and interpretation

On the chronological test set, the decision tree had an overall MAE of **94.6 Wh**, compared with **52.7 Wh** for the training-mean baseline. Across the **325 high-demand intervals** with actual usage of at least **210 Wh**, the tree had a lower MAE of **135.5 Wh** versus **259.2 Wh** for the baseline. The tree still underestimated usage in those intervals by **106.3 Wh on average**.

The tree uses the latest 20% of the data for testing. The KNN, MLP, and bagging notebooks use a randomized 70/30 split, so a head-to-head ranking requires evaluating the models on the same test rows.

Each point below represents a high-demand test interval. Points below the diagonal show underprediction. The vertical gap gives the size of the error in watt-hours.

In [ ]:
predictions = pd.read_csv(DEFAULT_OUTPUT_DIR / "predictions.csv")
peak_predictions = predictions.loc[predictions["high_demand"]]
upper_limit = peak_predictions[["actual_Wh", "tree_Wh"]].max().max()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(peak_predictions["actual_Wh"], peak_predictions["tree_Wh"], alpha=0.45)
ax.plot([0, upper_limit], [0, upper_limit], "--", color="gray", label="Perfect prediction")
ax.set(xlabel="Actual usage (Wh)", ylabel="Predicted usage (Wh)",
       title="Decision tree predictions during high demand")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()